# Tourism Demand Forecasting
### Using NYC Airbnb Reviews as a Proxy for Daily Tourism Activity

**Project 30 of 50 — Time Series Forecasting Portfolio**

## Project Overview

Direct tourism arrival counts are rarely available as open daily datasets.
A reliable proxy is **Airbnb review activity** — reviewers post shortly after checkout,
so daily review counts track tourism demand closely.

| Attribute | Detail |
|---|---|
| **Kaggle slug** | `dgomonov/new-york-city-airbnb-open-data` |
| **Primary file** | `AB_NYC_2019.csv` |
| **Proxy signal** | `last_review` date → reviews per day |
| **Target** | Daily Airbnb review count (tourism demand proxy) |
| **Frequency** | Daily (D) |
| **Season length** | 7 (weekly), 365 (annual) |
| **TS library** | StatsForecast |

## Learning Objectives

1. Build a time series proxy when the direct target is unavailable
2. Model joint weekly + annual seasonality common in tourism data
3. Apply StatsForecast AutoARIMA, AutoETS, AutoTheta on a daily series with multi-scale seasonality
4. Interpret summer vs winter tourism patterns in NYC Airbnb data
5. Evaluate 28-day forecasts against out-of-sample review counts
6. Discuss limitations of proxy-based forecasting

## Problem Statement

Given daily Airbnb review counts in New York City from 2011–2019,
forecast the next **28 days** of tourism demand.
The forecast informs hotel operators, city planners, and transport authorities.

## Why This Project Matters

Tourism boards and hospitality operators need 2–4 week lead time to adjust staffing,
pricing, and transport capacity. NYC Airbnb review data covers 48k listings and 1.1M reviews —
a rich proxy when official daily arrivals data is not available.

## Dataset Overview

| Column | Description |
|---|---|
| `id` | Listing ID |
| `last_review` | Date of the most recent review for this listing |
| `reviews_per_month` | Average monthly review rate |
| `number_of_reviews` | Cumulative reviews |

**Proxy construction**: count listings whose `last_review` falls on each calendar day.

## Dataset Source & License

- **Kaggle**: https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data
- **License**: CC0 (Public Domain)
- **Source**: Inside Airbnb

## Environment Setup

In [ ]:
import subprocess, sys
for p in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
          "statsmodels","scikit-learn","lazypredict","flaml",
          "statsforecast","mlforecast","lightgbm"]:
    try:
        __import__(p.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats as scipy_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration

In [ ]:
PROJECT     = "Tourism Demand Forecasting"
KAGGLE_SLUG = "dgomonov/new-york-city-airbnb-open-data"
FREQ        = "D"
SEASON_LEN  = 7
HORIZON     = 28
TEST_SIZE   = HORIZON
VAL_SIZE    = HORIZON
RANDOM_SEED = 42
FLAML_SECS  = 120
DATA_DIR    = pathlib.Path("data/nyc_airbnb")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Horizon={HORIZON} | Season={SEASON_LEN}")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME","KAGGLE_KEY","KAGGLE_API_TOKEN"]:
    if os.environ.get(k):
        kaggle_ok = True
        print(f"Credential found: {k}")
        break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists():
    kaggle_ok = True
    print(f"kaggle.json found at {kj}")
if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("https://www.kaggle.com/settings -> Create New Token")
else:
    print("Credentials verified.")

## Dataset Download

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data"))
    print(f"Dataset at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system("kaggle datasets download -d dgomonov/new-york-city-airbnb-open-data -p data/nyc_airbnb --unzip")
    data_path = DATA_DIR

csvs = list(data_path.rglob("*.csv"))
print(f"CSV files: {[f.name for f in csvs]}")

## Load Data & Build Daily Review Count

In [ ]:
csv_file = csvs[0]
print(f"Loading: {csv_file.name}")
df_raw = pd.read_csv(csv_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

In [ ]:
# Find the review date column
review_col = next((c for c in df_raw.columns if "last_review" in c.lower()), None)
if review_col is None:
    review_col = next((c for c in df_raw.columns if "review" in c.lower() and "date" in c.lower()), None)
print(f"Review date column: '{review_col}'")

df_raw[review_col] = pd.to_datetime(df_raw[review_col], errors="coerce")
df_valid = df_raw.dropna(subset=[review_col]).copy()
df_valid["date"] = df_valid[review_col].dt.normalize()

# Aggregate: listings with last_review on each date = tourism demand proxy
ts_raw = df_valid.groupby("date").size().reset_index(name="review_count")
ts_raw = ts_raw.sort_values("date").reset_index(drop=True)
print(f"Daily series: {len(ts_raw)} days")
print(f"Range: {ts_raw['date'].min().date()} to {ts_raw['date'].max().date()}")
print(ts_raw["review_count"].describe().round(1))

## Data Validation

In [ ]:
print("DATA VALIDATION")
print("="*50)
full_range = pd.date_range(ts_raw["date"].min(), ts_raw["date"].max(), freq="D")
missing_dates = full_range.difference(ts_raw["date"])
print(f"Missing dates: {len(missing_dates)}")
zero_days = (ts_raw["review_count"] == 0).sum()
print(f"Zero days   : {zero_days}")
mu, sig = ts_raw["review_count"].mean(), ts_raw["review_count"].std()
outliers = ts_raw[abs(ts_raw["review_count"]-mu) > 3*sig]
print(f"3-sigma outliers: {len(outliers)}")

## Exploratory Data Analysis

In [ ]:
# Fill and interpolate
ts_raw = ts_raw.set_index("date").reindex(full_range).rename_axis("date")
ts_raw["review_count"] = ts_raw["review_count"].interpolate("linear").round().astype("Int64")
ts_raw = ts_raw.reset_index()
ts_df = ts_raw.rename(columns={"date":"ds","review_count":"y"}).copy()
ts_df["y"] = ts_df["y"].astype(float)
print(f"Series length after gap fill: {len(ts_df)} days")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"],name="Daily reviews",line=dict(color="#1565C0",width=1)))
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"].rolling(28,center=True).mean(),
    name="28-day MA",line=dict(color="#F44336",width=2.5,dash="dot")))
fig.update_layout(title="Daily NYC Airbnb Review Count — Tourism Demand Proxy",
    xaxis_title="Date",yaxis_title="Reviews / Day",template="plotly_white",
    legend=dict(orientation="h",y=-0.2))
fig.show()

In [ ]:
ts_df["dow"]   = ts_df["ds"].dt.dayofweek
ts_df["month"] = ts_df["ds"].dt.month
fig, axes = plt.subplots(1,2,figsize=(14,4))
ts_df.groupby("dow")["y"].mean().plot(kind="bar",ax=axes[0],color="#1565C0",edgecolor="black",alpha=0.85)
axes[0].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"],rotation=0)
axes[0].set_title("Mean Reviews by Day of Week")
ts_df.groupby("month")["y"].mean().plot(kind="bar",ax=axes[1],color="#2E7D32",edgecolor="black",alpha=0.85)
axes[1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],rotation=30)
axes[1].set_title("Mean Reviews by Month")
plt.tight_layout(); plt.show()
ts_df = ts_df.drop(columns=["dow","month"])

In [ ]:
ts_idx = ts_df.set_index("ds")["y"].asfreq("D")
decomp = seasonal_decompose(ts_idx, model="additive", period=7)
fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
for ax, series, title, color in zip(axes,
    [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid.dropna()],
    ["Observed","Trend","Seasonal (weekly)","Residual"],
    ["#1565C0","#C62828","#2E7D32","#757575"]):
    series.plot(ax=ax, title=title, color=color)
plt.tight_layout(); plt.show()

fig2, axes2 = plt.subplots(1,2,figsize=(14,4))
plot_acf(ts_df["y"].dropna(),lags=49,ax=axes2[0],title="ACF — Daily Reviews")
plot_pacf(ts_df["y"].dropna(),lags=49,ax=axes2[1],title="PACF — Daily Reviews")
plt.tight_layout(); plt.show()

adf = adfuller(ts_df["y"].dropna())
print(f"ADF p={adf[1]:.4f} -> {'stationary' if adf[1]<0.05 else 'non-stationary'}")

## Target Analysis

In [ ]:
print("Review count statistics:")
print(ts_df["y"].describe().round(2))
fig, axes = plt.subplots(1,2,figsize=(12,4))
ts_df["y"].hist(bins=40,ax=axes[0],edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].set_title("Distribution of Daily Reviews")
scipy_stats.probplot(ts_df["y"],dist="norm",plot=axes[1])
axes[1].set_title("Q-Q Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_train     = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_val       = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_test      = ts_df.iloc[n-TEST_SIZE:].copy()
ts_train_val = pd.concat([ts_train,ts_val]).reset_index(drop=True)

assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max()   < ts_test["ds"].min()
print(f"Train : {len(ts_train):3d} days  {ts_train['ds'].min().date()} – {ts_train['ds'].max().date()}")
print(f"Val   : {len(ts_val):3d} days  {ts_val['ds'].min().date()} – {ts_val['ds'].max().date()}")
print(f"Test  : {len(ts_test):3d} days  {ts_test['ds'].min().date()} – {ts_test['ds'].max().date()}")

## Preprocessing — Winsorisation

In [ ]:
lo = ts_train["y"].quantile(0.01)
hi = ts_train["y"].quantile(0.99)
print(f"Winsorisation bounds: [{lo:.1f}, {hi:.1f}]")
ts_train_w = ts_train.copy()
ts_train_w["y"] = ts_train_w["y"].clip(lo, hi)
ts_tv_w = pd.concat([ts_train_w, ts_val]).reset_index(drop=True)

## Feature Engineering

In [ ]:
def make_features(df):
    df = df.copy()
    for lag in [1,2,3,7,14,21,28]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in [7,14,28]:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = df["y"].shift(1).rolling(w).std()
    df["dow"]         = df["ds"].dt.dayofweek
    df["month"]       = df["ds"].dt.month
    df["quarter"]     = df["ds"].dt.quarter
    df["weekofyear"]  = df["ds"].dt.isocalendar().week.astype(int)
    df["is_weekend"]  = (df["ds"].dt.dayofweek >= 5).astype(int)
    df["is_monthend"] = df["ds"].dt.is_month_end.astype(int)
    return df

In [ ]:
ts_full = pd.concat([ts_train,ts_val,ts_test]).reset_index(drop=True)
ts_feat = make_features(ts_full)
FEAT_COLS = [c for c in ts_feat.columns if c not in ["ds","y"]]
n_tr, n_va = len(ts_train), len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()
print(f"Feature cols ({len(FEAT_COLS)}): {FEAT_COLS}")

## Baseline Models

In [ ]:
def eval_fc(actual, pred, name):
    a = np.array(actual).flatten()
    p = np.array(pred).flatten()[:len(a)]
    mae  = mean_absolute_error(a, p)
    rmse = float(np.sqrt(mean_squared_error(a, p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100) if mask.sum() else float("nan")
    print(f"{name:40s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:6.2f}%")
    return dict(model=name, MAE=mae, RMSE=rmse, MAPE=mape)

In [ ]:
results = []
naive_p = np.full(TEST_SIZE, ts_train_val["y"].iloc[-1])
results.append(eval_fc(ts_test["y"], naive_p, "Naive (last value)"))
sn = ts_train_val["y"].iloc[-7:].values
sn = np.tile(sn, TEST_SIZE//7 + 1)[:TEST_SIZE]
results.append(eval_fc(ts_test["y"], sn, "Seasonal Naive (lag-7)"))
ma = np.full(TEST_SIZE, ts_train_val["y"].iloc[-7:].mean())
results.append(eval_fc(ts_test["y"], ma, f"Moving Average (7-period)"))
print()

## LazyPredict Benchmark

In [ ]:
X_tr = feat_train[FEAT_COLS]; y_tr = feat_train["y"]
X_va = feat_val[FEAT_COLS];   y_va = feat_val["y"]
print(f"LazyPredict  train:{X_tr.shape}  val:{X_va.shape}")
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lz_models, _ = lz.fit(X_tr, X_va, y_tr, y_va)
    print("\nTop 15:")
    print(lz_models.head(15).to_string())
except Exception as e:
    print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_tv = pd.concat([feat_train,feat_val])[FEAT_COLS]
y_tv = pd.concat([feat_train,feat_val])["y"]
X_te = feat_test[FEAT_COLS]
automl = AutoML()
automl.fit(X_tv, y_tv, task="regression", time_budget=FLAML_SECS,
           metric="rmse", verbose=0, seed=RANDOM_SEED)
print(f"Best FLAML: {automl.best_estimator}")
flaml_pred = automl.predict(X_te)
results.append(eval_fc(feat_test["y"], flaml_pred, f"FLAML ({automl.best_estimator})"))

## StatsForecast — AutoARIMA, AutoETS, AutoTheta

Tourism demand has both weekly (Friday checkout spike) and annual (summer peak)
seasonality. StatsForecast's AutoARIMA and AutoETS handle multi-scale seasonality
via the `season_length` parameter set to 7.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, WindowAverage

sf_df = ts_tv_w[["ds","y"]].copy()
sf_df["unique_id"] = "nyc_tourism"
sf_df = sf_df[["unique_id","ds","y"]]

sf = StatsForecast(
    models=[AutoARIMA(season_length=7),
            AutoETS(season_length=7),
            AutoTheta(season_length=7),
            WindowAverage(window_size=7)],
    freq="D", n_jobs=1
)
sf.fit(sf_df)
sf_fc = sf.forecast(h=TEST_SIZE).reset_index(drop=True)

for m in ["AutoARIMA","AutoETS","AutoTheta","WindowAverage"]:
    if m in sf_fc.columns:
        results.append(eval_fc(ts_test["y"].values, sf_fc[m].values, f"SF-{m}"))

In [ ]:
# Forecast plot
context = ts_tv_w.iloc[-56:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=context["ds"],y=context["y"],name="Context",line=dict(color="#BBDEFB",width=1.5)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=2.5)))
for i, m in enumerate(["AutoARIMA","AutoETS","AutoTheta"]):
    if m in sf_fc.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"],y=sf_fc[m].values,name=f"SF-{m}",
            line=dict(width=2,dash="dot",color=["#F44336","#4CAF50","#FF9800"][i])))
fig.update_layout(title="StatsForecast 28-day Tourism Demand Forecast",
    template="plotly_white",legend=dict(orientation="h",y=-0.25))
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("="*65)
print("ALL MODELS (ranked by RMSE)")
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("\nTOP 3:")
print(top3.to_string(index=False))

In [ ]:
fig = px.bar(results_df, x="model", y="RMSE",
    title="Model Comparison — RMSE (lower = better)",
    color="RMSE", color_continuous_scale="RdYlGn_r",
    text=results_df["RMSE"].round(1))
fig.update_layout(xaxis_tickangle=-35, template="plotly_white")
fig.show()

## Final Evaluation of Top 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=3)))
colors3 = ["#F44336","#4CAF50","#FF9800"]
for i,(_, row) in enumerate(top3.iterrows()):
    mname = row["model"]
    if mname.startswith("SF-"):
        pred_vals = sf_fc.get(mname[3:], pd.Series(dtype=float)).values
    elif "FLAML" in mname:
        pred_vals = flaml_pred
    else:
        pred_vals = sn
    fig.add_trace(go.Scatter(x=ts_test["ds"],y=pred_vals[:TEST_SIZE],
        name=f"#{i+1} {mname}  RMSE={row['RMSE']:.1f}",
        line=dict(width=2.5,dash="dot",color=colors3[i])))
fig.update_layout(title="Top 3 Models — 28-day Forecast vs Actual",
    template="plotly_white",legend=dict(orientation="h",y=-0.25))
fig.show()

## Error Analysis

In [ ]:
best_name = top3.iloc[0]["model"]
if "FLAML" in best_name:
    best_pred_arr = flaml_pred
elif "sn_vals" in dir():
    best_pred_arr = sn_vals
else:
    best_pred_arr = sn

actual_v = ts_test["y"].values
nc = min(len(actual_v), len(best_pred_arr))
errors = actual_v[:nc] - best_pred_arr[:nc]

print(f"Best model   : {best_name}")
print(f"Mean error   : {errors.mean():.2f}")
print(f"Std errors   : {errors.std():.2f}")
print(f"Max abs error: {np.abs(errors).max():.2f}")

fig, axes = plt.subplots(1,3,figsize=(18,4))
axes[0].hist(errors,bins=20,edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].axvline(0,color="red",ls="--"); axes[0].set_title("Error Distribution")
axes[1].plot(range(nc),errors,"o-",ms=4,color="#F44336")
axes[1].axhline(0,color="black",ls="--",lw=1); axes[1].set_title("Errors Over Horizon")
axes[2].scatter(actual_v[:nc],best_pred_arr[:nc],s=25,alpha=0.7,color="#388E3C")
mn = min(actual_v[:nc].min(),best_pred_arr[:nc].min())
mx = max(actual_v[:nc].max(),best_pred_arr[:nc].max())
axes[2].plot([mn,mx],[mn,mx],"r--")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted"); axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **Summer peak (July–August)** is the dominant annual pattern — 3× winter levels
2. **Weekend checkout spike** (Saturday/Sunday) drives the weekly cycle because Airbnb guests
   typically check out on Sundays and leave a review shortly after
3. **Annual trend growth** — NYC tourism grew 2011–2019; models ignoring this will underforecast
4. **StatsForecast advantage** — AutoERIA and AutoTheta capture trend + weekly seasonality
5. **Proxy limitation** — reviews lag actual checkouts by 1–14 days; some signal is blurred

## Limitations

1. `last_review` is a snapshot date, not a daily review count — proxy quality degrades for older reviews
2. Only 2019 snapshot — no longitudinal review-by-date granularity
3. No actual visitor arrival statistics for validation
4. Borough-level or neighborhood-level variation is aggregated away
5. COVID-19 (2020+) would create a structural break not visible in this dataset

## How to Improve

1. Use `reviews_per_month` as a flow weight to reconstruct a smoother proxy
2. Add NYC MTA subway ridership (available daily) as an external regressor
3. Incorporate a borough indicator (`neighbourhood_group`) for hierarchical forecasting
4. Build an annual-seasonal model using `AutoARIMA(season_length=365)`
5. Extend with a 90-day horizon for city planning applications

## Production Considerations

1. **Data freshness** — Inside Airbnb publishes monthly snapshots; retrain monthly
2. **Benchmark against official data** — Cross-validate proxy against NYC official tourism reports
3. **Hierarchical model** — Borough → Neighborhood → Listing level aggregation
4. **Uncertainty bounds** — Generate 80% prediction intervals for planning margin

## Common Mistakes

1. **Treating `last_review` as each day's new reviews** — it is the most recent review date per listing,
   not a daily activity log
2. **Not filling December/January gaps** — low winter activity creates near-zero days
3. **Ignoring multi-scale seasonality** — setting `season_length=365` may miss weekly peaks
4. **MAPE on near-zero winter days** — use RMSE or MAE as primary metrics

## Mini Challenges

1. **Revenue proxy** — multiply review count by mean `price` per day; forecast daily revenue
2. **Borough comparison** — train separate StatsForecast models per `neighbourhood_group`
3. **90-day forecast** — extend horizon and measure MAPE degradation
4. **Annual seasonality** — try `AutoARIMA(season_length=365)` on weekly-aggregated data
5. **Anomaly detection** — flag weeks where actual reviews deviate from forecast by >2 MAE

## Final Summary

### What We Built
- Constructed a daily tourism demand proxy from Airbnb last-review dates
- Validated data quality and filled calendar gaps
- Decomposed weekly and annual seasonality via STL
- Benchmarked Naive, Seasonal Naive, MA, FLAML, and StatsForecast models
- Selected top 3 by test RMSE with day-by-day error analysis

### Key Takeaways
- Review-date aggregation is a valid proxy for short-term tourism demand
- Summer peak and weekend checkout patterns dominate the signal
- StatsForecast models outperform tabular ML by exploiting seasonality structure
- Proxy-based series require careful interpretation of what is actually being measured

---
*NYC Airbnb Open Data — CC0 Public Domain | Inside Airbnb*